In [2]:
import os
import sys
from getpass import getpass
from pathlib import Path
from typing import Any, Dict, List, Tuple, Type
import requests
import subprocess
import time
from pydantic import BaseModel
import json

In [3]:
token = getpass("GitHub PAT: ")
os.environ["GITHUB_TOKEN"] = token

!git clone https://$GITHUB_TOKEN@github.com/Hanh3103/BA_USDependency.git

%cd /content/BA_USDependency

!git checkout Tool

current_dir = os.getcwd()
repo_root = current_dir
while not os.path.exists(os.path.join(repo_root, "src")):
    parent = os.path.dirname(repo_root)
    if parent == repo_root:  # Reached system root without finding it
        repo_root = os.path.abspath(os.path.join(current_dir, "..", "..", ".."))
        break
    repo_root = parent

src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)



GitHub PAT: ··········
Cloning into 'BA_USDependency'...
remote: Enumerating objects: 1403, done.
remote: Counting objects: 100% (270/270), done.
remote: Compressing objects: 100% (198/198), done.
remote: Total 1403 (delta 123), reused 184 (delta 65), pack-reused 1133 (from 1)
Receiving objects: 100% (1403/1403), 2.59 MiB | 6.62 MiB/s, done.
Resolving deltas: 100% (703/703), done.
/content/BA_USDependency
branch 'Tool' set up to track 'origin/Tool'.
Switched to a new branch 'Tool'


In [ ]:
!git pull

In [5]:
!git config --global user.email "thiminhhanhnguyen.95@gmail.com"
!git config --global user.name "Hanh3103"

In [4]:
print("Repo Root:", repo_root)
print("App Path added to sys.path:", src_path)

Repo Root: /content/BA_USDependency
App Path added to sys.path: /content/BA_USDependency/src


In [6]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 8.0 MB/s eta 0:00:00
  Attempting uninstall: uvicorn
    Found existing installation: uvicorn 0.52.4
    Uninstalling uvicorn-0.52.4:
      Successfully uninstalled uvicorn-0.52.4
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.5
    Uninstalling pydantic_core-2.46.5:
      Successfully uninstalled pydantic_core-2.46.5
  Attempting uninstall: star

In [7]:
env_content = """OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_API_KEY=ollama
OLLAMA_MODELS=qwen3:8b,llama3.1:8b
OLLAMA_TEMPERATURE=0.0
OLLAMA_TOP_P=0.1
OLLAMA_NUM_PREDICT=2048
OLLAMA_NUM_CTX=8192
OLLAMA_KEEP_ALIVE=20m
OLLAMA_THINK=false
OLLAMA_FORMAT=json
"""

env_target = os.path.join(repo_root, ".env")
with open(env_target, "w") as f:
    f.write(env_content.strip())

print(f"Wrote .env to: {env_target}")

Wrote .env to: /content/BA_USDependency/.env


In [8]:
from config import settings
from stages.dependency_pipeline_text import run_dependency_pipeline_text
from stages.dependency_pipeline_annotated import run_dependency_pipeline
from schemas.dependency.dependency_schema import AnnotatedDependencyResult, TextDependencyResult

ENV path: /content/BA_USDependency/.env
ENV exists: True
ENV values: OrderedDict({'OLLAMA_BASE_URL': 'http://localhost:11434', 'OLLAMA_API_KEY': 'ollama', 'OLLAMA_MODELS': 'qwen3:8b,llama3.1:8b', 'OLLAMA_TEMPERATURE': '0.0', 'OLLAMA_TOP_P': '0.1', 'OLLAMA_NUM_PREDICT': '2048', 'OLLAMA_NUM_CTX': '8192', 'OLLAMA_KEEP_ALIVE': '20m', 'OLLAMA_THINK': 'false', 'OLLAMA_FORMAT': 'json'})


In [9]:
!apt-get update -qq
!apt-get install -y zstd

# 1. Install Ollama binary
!curl -fsSL https://ollama.com/install.sh | sh

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["OLLAMA_KEEP_ALIVE"] = "-1"      # Keep models pinned in VRAM
env["OLLAMA_NUM_PARALLEL"] = "4"     # Enable parallel pipeline processing

ollama_process = subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(5)  # Wait for server startup

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 71 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (391 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zs

In [10]:
for model in settings.models:
    model = model.strip()
    print(f"Pulling {model}...")
    result = subprocess.run(["ollama", "pull", model], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"FAILED to pull {model}:\n{result.stderr}")
    else:
        print(f"OK: {model}")

Pulling qwen3:8b...
OK: qwen3:8b
Pulling llama3.1:8b...
OK: llama3.1:8b


In [ ]:
!ollama rm llama3.1:8b

deleted 'llama3.1:8b'


In [ ]:
!ollama pull llama3.1:8b

In [11]:
DATASET_PATH_GT = os.path.join("eval_with_colab","annotation","baseline", "20US_baseline.json")
if os.path.exists(DATASET_PATH_GT):
    print("Annotated story file found successfully!")
else:
    print("Error: Annotated story  file not found. Check path.")

Annotated story file found successfully!


In [12]:
DATASET_PATH_LLM_ANNOTATED = os.path.join("eval_with_colab", "annotation", "prediction", "qwen_pred_ann.json")

if os.path.exists(DATASET_PATH_LLM_ANNOTATED):
    print("LLM-annotated story file found successfully!")
else:
    print("Error: LLM-annotated story file not found. Check path.")

LLM-annotated story file found successfully!


In [13]:
CONDITION_NAMES = ["text_gt", "annotated_gt", "annotated_llm"]

RESULTS_DIR = Path(repo_root) / "eval_with_colab" / "dependency_analysis" / "pipeline_outputs"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ERRORS_FILE = RESULTS_DIR / "errors.json"
print(f"Results will be saved to {RESULTS_DIR}")
print(f"Errors will be saved to {ERRORS_FILE}")

Results will be saved to /content/BA_USDependency/eval_with_colab/dependency_analysis/pipeline_outputs
Errors will be saved to /content/BA_USDependency/eval_with_colab/dependency_analysis/pipeline_outputs/errors.json


**Helper function**

In [14]:
def unload_ollama_model(model_name: str) -> None:
    """Evict current model from GPU memory to prevent VRAM OOM on model switch."""
    try:
        requests.post(
            f"{settings.ollama_base_url}/api/generate",
            json={"model": model_name, "keep_alive": 0},
            timeout=5,
        )
    except Exception:
        pass

In [15]:
def resolve_condition_config(condition_name: str) -> Tuple[Path, Any, Type[BaseModel]]:
    """Maps evaluation condition to dataset path, pipeline function, and Pydantic schema."""
    if condition_name == "text_gt":
        return DATASET_PATH_GT, run_dependency_pipeline_text, TextDependencyResult
    elif condition_name == "annotated_gt":
        return DATASET_PATH_GT, run_dependency_pipeline, AnnotatedDependencyResult
    elif condition_name == "annotated_llm":
        return DATASET_PATH_LLM_ANNOTATED, run_dependency_pipeline, AnnotatedDependencyResult
    else:
        raise ValueError(f"Unknown condition: {condition_name}")

In [16]:
def save_results_to_cache(results_file, meta_file, results, runtime_seconds, failed_pairs=None, attempts_by_pair=None, dropped=None, kept_ids=None):
    result_dicts = [r.model_dump() for r in results]
    results_file.write_text(json.dumps(result_dicts, indent=2), encoding="utf-8")

    n_pairs = len(results)
    meta_data = {
        "runtime_seconds": runtime_seconds,
        "n_pairs": n_pairs,
        "seconds_per_pair": (runtime_seconds / n_pairs) if n_pairs > 0 else None,
        "failed_pairs": failed_pairs or [],
        "attempts_by_pair": {f"{a}|{b}": n for (a, b), n in (attempts_by_pair or {}).items()},
        "n_kept_stories": len(kept_ids),
        "kept_ids": kept_ids,
        "ill_formed_count": len(dropped or []),
        "ill_formed_stories": [
            {"id": r.id, "text": r.Text, "reason": r.well_formed_reason}
            for r in (dropped or [])
        ],
    }
    meta_file.write_text(json.dumps(meta_data, indent=2), encoding="utf-8")

In [17]:
def save_errors_to_disk(errors: List[Dict[str, Any]]) -> None:
    """Persists error logs instantly so failures survive notebook runtime disconnects."""
    if errors:
        ERRORS_FILE.write_text(json.dumps(errors, indent=2), encoding="utf-8")

In [18]:
def run_single_condition(model_name: str, condition_name: str, max_retries: int = 2):
    """Executes a single pipeline condition for a specific model."""
    # 1. Handle VRAM eviction and update active model
    if settings.models and settings.models[0] != model_name:
        unload_ollama_model(settings.models[0])
    settings.models[0] = model_name

    # 2. Resolve configuration mapping
    dataset_path, pipeline_fn, schema_class = resolve_condition_config(condition_name)

    if not Path(dataset_path).exists():
        raise FileNotFoundError(f"Dataset path does not exist: {dataset_path}")

    print(f"== Running {model_name} | Condition: '{condition_name}' ==")

    # 3. Execute pipeline (retries happen inside detect_dependency / detect_dependency_text per pair)
    start_time = time.perf_counter()
    results, kept_ids,dropped, failed_pairs, attempts_by_pair = pipeline_fn(
        data_set_path=dataset_path, model=model_name, max_retries=max_retries
    )
    runtime_seconds = time.perf_counter() - start_time

    print(f"Finished {len(results)} pairs in {runtime_seconds:.1f} seconds.")
    if failed_pairs:
        print(f"  {len(failed_pairs)} pairs failed after {max_retries} attempts: {failed_pairs}\n")
    else:
        print()

    return results, runtime_seconds, kept_ids, dropped, failed_pairs, attempts_by_pair

In [ ]:
!pkill ollama
time.sleep(2)
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["OLLAMA_KEEP_ALIVE"] = "-1"
env["OLLAMA_NUM_PARALLEL"] = "4"
ollama_process = subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(5)

!curl -s http://localhost:11434/api/tags
!ollama list

**LLAMA-EXECUTION**

In [ ]:
llama_text_res, llama_text_time, llama_text_ids,llama_text_dropped, llama_text_failed, llama_text_attempts = run_single_condition(
    model_name="llama3.1:8b",
    condition_name="text_gt"
)

In [ ]:
llama_ann_corr_res, llama_ann_corr_time, llama_ann_corr_ids,llama_ann_corr_dropped, llama_ann_corr_failed, llama_ann_corr_attempts = run_single_condition(
    model_name="llama3.1:8b",
    condition_name="annotated_gt"
)

== Running llama3.1:8b | Condition: 'annotated_gt' ==
Kept 16 well-formed stories, dropped 4 ill-formed stories.
Dropped story As a new user, I want to sign up using my email or an existing Google/Apple account, so that my saved spots and trips are securely stored and synced across my devices.: BenefitAction/BenefitEntity present but no targets pair connects them
Dropped story As a traveler, I want to remove a saved spot from my lists, so that my lists and map stay clean and only contain places I still intend to visit.: BenefitAction/BenefitEntity present but no targets pair connects them
Dropped story As a traveler, I want to specify the start date, end date, or total number of days for a trip, so that the AI itinerary matches the exact length of my stay.: BenefitAction/BenefitEntity present but no targets pair connects them
Dropped story As a traveler, I want to permanently delete a trip I no longer need, so that my trip list remains uncluttered and only shows active or relevant plan

**QWEN-EXECUTION**

In [ ]:
qwen_text_res, qwen_text_time, qwen_text_ids, qwen_text_dropped, qwen_text_failed, qwen_text_attempts = run_single_condition(
    model_name="qwen3:8b",
    condition_name="text_gt"
)

In [ ]:
qwen_ann_corr_res, qwen_ann_corr_time, qwen_ann_corr_ids, qwen_ann_corr_dropped, qwen_ann_corr_failed, qwen_ann_corr_attempts = run_single_condition(
    model_name="qwen3:8b",
    condition_name="annotated_gt"
)

== Running qwen3:8b | Condition: 'annotated_gt' ==
Kept 16 well-formed stories, dropped 4 ill-formed stories.
Dropped story As a new user, I want to sign up using my email or an existing Google/Apple account, so that my saved spots and trips are securely stored and synced across my devices.: BenefitAction/BenefitEntity present but no targets pair connects them
Dropped story As a traveler, I want to remove a saved spot from my lists, so that my lists and map stay clean and only contain places I still intend to visit.: BenefitAction/BenefitEntity present but no targets pair connects them
Dropped story As a traveler, I want to specify the start date, end date, or total number of days for a trip, so that the AI itinerary matches the exact length of my stay.: BenefitAction/BenefitEntity present but no targets pair connects them
Dropped story As a traveler, I want to permanently delete a trip I no longer need, so that my trip list remains uncluttered and only shows active or relevant plans.:

In [19]:
qwen_ann_llm_res, qwen_ann_llm_time, qwen_ann_llm_ids, qwen_ann_llm_dropped, qwen_ann_llm_failed, qwen_ann_llm_attempts = run_single_condition(
    model_name="qwen3:8b",
    condition_name="annotated_llm"
)

== Running qwen3:8b | Condition: 'annotated_llm' ==
Kept 15 well-formed stories, dropped 5 ill-formed stories.
Dropped story As a user, I want to walk through a short, guided tutorial that demonstrates how to import a spot, create a list, and generate an itinerary, so that I can understand the app's core workflow without reading external documentation.: BenefitAction/BenefitEntity present but no targets pair connects them
Dropped story As a traveler, I want to open the original TikTok or Instagram video that was used to save a spot, so that I can rewatch the creator's tips and context before visiting.: BenefitAction/BenefitEntity present but no targets pair connects them
Dropped story As a traveler, I want to remove a saved spot from my lists, so that my lists and map stay clean and only contain places I still intend to visit.: BenefitAction/BenefitEntity present but no targets pair connects them
Dropped story As a traveler, I want to generate a read-only web link for my itinerary, so 

**SAVE to FILES**

In [ ]:
conditions = [
    ("llama3.1:8b", "annotated_gt", llama_ann_corr_res, llama_ann_corr_time, llama_ann_corr_ids, llama_ann_corr_dropped, llama_ann_corr_failed, llama_ann_corr_attempts),
    ("qwen3:8b", "annotated_gt", qwen_ann_corr_res, qwen_ann_corr_time, qwen_ann_corr_ids, qwen_ann_corr_dropped, qwen_ann_corr_failed, qwen_ann_corr_attempts),
]

for model_name, condition_name, results, runtime, kept_ids, dropped, failed_pairs, attempts_by_pair in conditions:
    safe_model = model_name.replace(":", "_")
    results_file = RESULTS_DIR / f"{safe_model}_{condition_name}_results.json"
    meta_file = RESULTS_DIR / f"{safe_model}_{condition_name}_meta.json"
    save_results_to_cache(results_file, meta_file, results, runtime, failed_pairs, attempts_by_pair, dropped, list(kept_ids))
    print(f"Cached: {results_file.name}")

Cached: llama3.1_8b_annotated_gt_results.json
Cached: qwen3_8b_annotated_gt_results.json


In [ ]:
conditions = [
    ("llama3.1:8b", "text_gt", llama_text_res, llama_text_time, llama_text_ids, llama_text_dropped, llama_text_failed, llama_text_attempts),
    ("llama3.1:8b", "annotated_gt", llama_ann_corr_res, llama_ann_corr_time, llama_ann_corr_ids, llama_ann_corr_dropped, llama_ann_corr_failed, llama_ann_corr_attempts),
    ("qwen3:8b", "text_gt", qwen_text_res, qwen_text_time, qwen_text_ids, qwen_text_dropped, qwen_text_failed, qwen_text_attempts),
    ("qwen3:8b", "annotated_gt", qwen_ann_corr_res, qwen_ann_corr_time, qwen_ann_corr_ids, qwen_ann_corr_dropped, qwen_ann_corr_failed, qwen_ann_corr_attempts),
]

for model_name, condition_name, results, runtime, kept_ids, dropped, failed_pairs, attempts_by_pair in conditions:
    safe_model = model_name.replace(":", "_")
    results_file = RESULTS_DIR / f"{safe_model}_{condition_name}_results.json"
    meta_file = RESULTS_DIR / f"{safe_model}_{condition_name}_meta.json"
    save_results_to_cache(results_file, meta_file, results, runtime, failed_pairs, attempts_by_pair, dropped, list(kept_ids))
    print(f"Cached: {results_file.name}")

NameError: name 'llama_text_res' is not defined

In [20]:
conditions = [
    ("qwen3:8b", "annotated_llm", qwen_ann_llm_res, qwen_ann_llm_time, qwen_ann_llm_ids, qwen_ann_llm_dropped, qwen_ann_llm_failed, qwen_ann_llm_attempts),
]

for model_name, condition_name, results, runtime, kept_ids, dropped, failed_pairs, attempts_by_pair in conditions:
    safe_model = model_name.replace(":", "_")
    results_file = RESULTS_DIR / f"{safe_model}_{condition_name}_results.json"
    meta_file = RESULTS_DIR / f"{safe_model}_{condition_name}_meta.json"
    save_results_to_cache(results_file, meta_file, results, runtime, failed_pairs, attempts_by_pair, dropped, list(kept_ids))
    print(f"Cached: {results_file.name}")

Cached: qwen3_8b_annotated_llm_results.json


In [21]:
!git add .
!git commit -m "rerun dependency analysis ouput llm"
!git push https://$GITHUB_TOKEN@github.com/Hanh3103/BA_USDependency.git Tool

[Tool 5cb2f2b] rerun dependency analysis ouput llm
 2 files changed, 426 insertions(+), 511 deletions(-)
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 2.04 KiB | 1.02 MiB/s, done.
Total 7 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 5 local objects.
To https://github.com/Hanh3103/BA_USDependency.git
   f7c46fb..5cb2f2b  Tool -> Tool
